In [ ]:
import numpy as np
from astropy.io import fits
from astropy.table import Table
from datetime import datetime
import copy
import collections
from astropy.cosmology import Planck13  # Planck (2014) Paper XVI, Table 5
import time 
import pymangle

import gc
gc.enable()

import sys
sys.path.append('..') 

from utils import Vcom, get_comoving_distance_h, c, z_min
from data_stat_func import weighted_wprp, weighted_wprp_theory
from data_stat_func import wprp

from halotools.mock_observables import apply_zspace_distortion
from halotools.mock_observables import return_xyz_formatted_array
import halotools.mock_observables as htmo

import Corrfunc
from Corrfunc.utils import convert_rp_pi_counts_to_wp
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

from Ng8_func import return_box_rsd_pos

In [ ]:
import Corrfunc
Corrfunc.__version__

## $w_p(r_p)$

In [ ]:
ccmd_period = 1000 # h^-1Mpc
rp_bins_wp   = np.logspace(-0.8, 1.6, 13)
rp_bins_wp_x = (rp_bins_wp[1:] + rp_bins_wp[:-1]) / 2

from scipy import constants
# Speed of light in km/s
c = constants.c/1e3

# Sample selection criteria from Zehavi 2011
z_min = 0.02
z_max_arr   = np.array((       47650, 31900, 19250, 12500  ))/ c
Mr_thr_arr  = np.array((-22.0, -21.0, -20.0, -19.0, -18.0, ))
Mr_thr_list = [ 'Mr_22_21', 'Mr_21_20', 'Mr_20_19', 'Mr_19_18', ]
Mr_thr_text = [ r'$M_r: [-22, -21]$', r'$M_r: [-21, -20]$', r'$M_r: [-20, -19]$', r'$M_r: [-19, -18]$', ]

### randoms

In [ ]:
# # /home/wenlk/AssemblyBias/code/lss_combmask.dr72.fits

# window   = pymangle.Mangle('/home/wenlk/AssemblyBias/code/window_pix.dr72bright1.ply')
# mask     = pymangle.Mangle('/home/wenlk/AssemblyBias/code/mask_pix.dr72bright1.ply')
# combmask = pymangle.Mangle('/home/wenlk/AssemblyBias/code/lss_combmask_pix.dr72.ply')
# combmask_info  = fits.open('/home/wenlk/AssemblyBias/code/lss_combmask.dr72.fits')[1].data
# mangle   = [window, mask, combmask, combmask_info]

In [ ]:
# ### generate wp random sample for ccmd
# N       = 12000000
# ra      = np.random.random(N) * 360.0
# sin_dec = np.random.random(N) * 2 - 1
# dec     = np.arcsin(sin_dec, ) * 180.0 / np.pi

# polyid   = combmask.polyid(ra, dec)
# areamask = ((polyid != -1) & (combmask_info['FGOTMAIN'][polyid] >= 0.8) &
#              window.contains(ra, dec) & np.logical_not(mask.contains(ra, dec)))

# ra    = ra[areamask]
# dec   = dec[areamask]
# ### CCMD 32 Quads
# quads = ['Quad0', 'Quad1', 'Quad2', 'Quad3', ]
# vars  = ['Observer_1', 'Observer_2', 'Observer_3', 'Observer_4', 'Observer_5', 'Observer_6', 'Observer_7', 'Observer_8']

# for var in vars:
#     for quad in quads:
#         data = fits.open('../CCMD_main_processed/CCMD_32Quads_processed/CCMD_' + var + '_' + quad + '_processed_catalog.fits')[1].data 
        
#         for i in range(4):
#             Mr_z_mask  = np.logical_and(np.logical_and( data['Mr'] > Mr_thr_arr[i], data['Mr'] < Mr_thr_arr[i+1]),
#                                         np.logical_and( data['z'] >= z_min        , data['z' ] < z_max_arr[i]   ))
#             ccmdz = np.array(data['z'][Mr_z_mask])
#             randz = np.tile(ccmdz, len(ra) // len(ccmdz) + 1 )[:len(ra)]
#             np.savez('./CCMD_32Quads_Mr_bins_Mrp_randoms/CCMD_' + var + '_' + quad  + '_' + Mr_thr_list[i] + '_randoms',
#                       randoms = np.vstack((ra, dec, randz)).T) 

#     print('i:',i)
#     print('len(ra):', len(ra))
#     print('len(randz):',len(randz))

### stat = n_g, Marked_wp

In [ ]:
### CCMD 32 Quads
# quads = ['Quad0', 'Quad1', 'Quad2', 'Quad3', ]
# vars  = ['Observer_1', 'Observer_2', 'Observer_3', 'Observer_4', 'Observer_5', 'Observer_6', 'Observer_7', 'Observer_8']

quads = ['Quad0']
vars  = ['Observer_1']

print('CCMD')

for var in vars:
    for quad in quads:
        data = fits.open('../CCMD_main_processed/CCMD_32Quads_processed/CCMD_' + var + '_' + quad + '_processed_catalog.fits')[1].data 
        output_dict = collections.defaultdict(list)
    #########################################################################################################
        for j in range(4):
            stat = []
            if j == 3:
                pi_max = 40.
            else :
                pi_max = 60.
        ### Mr mask is related to the z_max #################################################################
            Mr_z_mask = np.logical_and( np.logical_and(Mr_thr_arr[j] < data['Mr'], data['Mr'] < Mr_thr_arr[j+1]),
                                        np.logical_and(data['z']    >= z_min     , data['z']  < z_max_arr[j]   ))
            gmr_notNaN_mask = np.logical_not(np.isnan(data['gmr']))
            data_thr   = data[Mr_z_mask & gmr_notNaN_mask]
        ### n_gal ###########################################################################################
            N_gal      = np.sum(Mr_z_mask & gmr_notNaN_mask)
            stat.append(np.array([N_gal / Vcom(z_max_arr[j])])) 
        ### Mrp #############################################################################################
            coords_thr = np.array([ np.array(data_thr['ra']),
                                    np.array(data_thr['dec']),    
                                    np.array(data_thr['z'])]).T
            gmr_thr    = data_thr['gmr']
            gmr_thr_ratio  = gmr_thr / np.mean(gmr_thr)# weights = gmr_thr_ratio

            # print('Mr:[%.2f,%.2f], gmr_mean:%.2f'%(Mr_thr_arr[j], Mr_thr_arr[j+1], np.mean(gmr_thr)))

            coords_rand_thr = np.load('/share/wenlk/AssemblyBias/CCMD/CCMD_32Quads_data/CCMD_32Quads_Mr_bins_Mrp_randoms/CCMD_' 
            # coords_rand_thr = np.load('./CCMD_32Quads_Mr_bins_Mrp_randoms/CCMD_' 
                                      + var + '_' + quad  + '_' + Mr_thr_list[j] + '_randoms' + '.npz')['randoms']

            coords_thr = coords_thr.astype(np.float64)
            gmr_thr_ratio = gmr_thr_ratio.astype(np.float64)
            coords_rand_thr = coords_rand_thr.astype(np.float64) # one of them is float32, make sure all to be float64

            wp          = wprp(         gal = coords_thr, random = coords_rand_thr, pi_max = pi_max, rp_bins = rp_bins_wp)
            Weighted_wp = weighted_wprp(gal = coords_thr, gal_weights = gmr_thr_ratio, 
                                        random = coords_rand_thr, pi_max = pi_max, rp_bins = rp_bins_wp)
            Marked_wp   = (1 + Weighted_wp/rp_bins_wp_x) / (1 + wp/rp_bins_wp_x) # Mrp = (1 + Wp/rp) / (1 + wp/rp)
            stat.append(Marked_wp)
            
            print(Marked_wp)

            # output_dict['stat_' + Mr_thr_list[j] ] = np.array([item for sublist in stat for item in sublist])
            
        # np.savez('./CCMD_32Quads_Mr_bins_Mrp/CCMD_' + var + '_' + quad + '_Mr_bins_Mrp_pimax60_60_60_40.npz', **output_dict)

In [ ]:
# ### CCMD 32 Quads cov
# quads = ['Quad0', 'Quad1', 'Quad2', 'Quad3', ]
# vars  = ['Observer_1', 'Observer_2', 'Observer_3', 'Observer_4', 'Observer_5', 'Observer_6', 'Observer_7', 'Observer_8']

# output_dict_stat = collections.defaultdict(list)
# output_dict_cov  = collections.defaultdict(list)

# for j in range(4): # Mr, z
#     data  = []
#     for var in vars:
#         for quad in quads:
#             temp = np.load('./CCMD_32Quads_Mr_bins_Mrp/CCMD_' + var + '_' + quad + '_Mr_bins_Mrp_pimax60_60_60_40.npz')
#             data.append(temp['stat_' + Mr_thr_list[j]])

#     data  = np.array(data)

#     stat = []
#     cov  = []
 
#     original_statistics = np.mean(data, axis=0)
#     cov_matrix          = np.cov(data, rowvar=False)  # rowvar=False, column for variants

#     stat.append(original_statistics)
#     cov.append(cov_matrix)

#     output_dict_stat['stat_' + Mr_thr_list[j] ] = np.array([item for sublist in stat for item in sublist])
#     output_dict_cov[ 'cov_'  + Mr_thr_list[j] ] = np.array([item for sublist in cov for item in sublist])

#     np.savez('./CCMD_32Quads_Mr_bins_Mrp_pimax_60_60_60_40_stat_mean.npz', **output_dict_stat)
#     np.savez('./CCMD_32Quads_Mr_bins_Mrp_pimax_60_60_60_40_cov.npz', **output_dict_cov)

In [ ]:
# ccmd_stat = np.load('./CCMD_32Quads_Mr_bins_Mrp_pimax_60_60_60_40_stat_mean.npz')
# ccmd_cov_matrix = np.load('./CCMD_32Quads_Mr_bins_Mrp_pimax_60_60_60_40_cov.npz')
# i = 3
# j = 3
# input_stat, corr_matrix, stdv = cov_to_corr(ccmd_stat['stat_' + Mr_thr_list[j] ], 
#                                             ccmd_cov_matrix['cov_' + Mr_thr_list[j] ])
# # Plot Correlation Coviriance Matrix
# plt.figure(figsize=(8, 6))
# plt.imshow(corr_matrix, cmap='coolwarm', interpolation='nearest')
# plt.colorbar(label='Covariance')
# plt.title('Corr' + Mr_thr_list[j], fontsize = 16)
# plt.show()

In [ ]:
### CCMD 20Vars box, rsd
vars = ['Var01', 'Var02', 'Var03', 'Var04', 'Var05', 'Var06', 'Var07', 'Var08', 'Var09', 'Var10', 
        'Var11', 'Var12', 'Var13', 'Var14', 'Var15', 'Var16', 'Var17', 'Var18', 'Var19', 'Var20', ]

for var in vars:
    data = fits.open("/share/hjxu/ccmdmock/Vars/" + var + "/CCMD_Mock_" + var + "_GalHaloInfo.fits")[1].data
    output_dict = collections.defaultdict(list)
#########################################################################################################
    for j in range(4):
        stat = []
        if j == 3:
            pi_max = 40.
        else :
            pi_max = 60.
    ### Mr mask is related to the z_max #################################################################
        Mr_mask = np.logical_and(Mr_thr_arr[j] < data['Mr'], data['Mr'] < Mr_thr_arr[j+1])
        gmr_notNaN_mask = np.logical_not(np.isnan(data['gmr']))
        data_thr   = data[Mr_mask & gmr_notNaN_mask]
    ### n_gal ###########################################################################################
        N_gal      = np.sum(Mr_mask & gmr_notNaN_mask)
        stat.append(np.array([N_gal / Vcom(z_max_arr[j])])) 
    ### Mrp #############################################################################################
        coords_thr = return_box_rsd_pos(data_thr['x'], data_thr['y'], data_thr['z'], data_thr['vz'],)
        gmr_thr    = data_thr['gmr']
        gmr_thr_ratio  = gmr_thr / np.mean(gmr_thr)# weights = gmr_thr_ratio

        NR = 10*len(data_thr['x'])
        R1_X = np.random.rand(NR) * 1000
        R1_Y = np.random.rand(NR) * 1000
        R1_Z = np.random.rand(NR) * 1000
        coords_rand_thr = np.array([R1_X, R1_Y, R1_Z]).T

        coords_thr = coords_thr.astype(np.float64)
        gmr_thr_ratio = gmr_thr_ratio.astype(np.float64)
        coords_rand_thr = coords_rand_thr.astype(np.float64) # one of them is float32, make sure all to be float64

        wp          = htmo.wp(coords_thr, rp_bins_wp, pi_max, period = ccmd_period, num_threads = 50)
        Weighted_wp = weighted_wprp_theory( gal = coords_thr, gal_weights = gmr_thr_ratio, random = coords_rand_thr, 
                                            pi_max = pi_max, rp_bins = rp_bins_wp, boxsize = ccmd_period)
        Marked_wp   = (1 + Weighted_wp/rp_bins_wp_x) / (1 + wp/rp_bins_wp_x) # Mrp = (1 + Wp/rp) / (1 + wp/rp)
        stat.append(Marked_wp)
        
        output_dict['stat_' + Mr_thr_list[j] ] = np.array([item for sublist in stat for item in sublist])
        
    np.savez('./CCMD_20Vars_Mrp/CCMD_' + var  + '_box_rsd_Mr_bins_Mrp_pimax60_60_60_40.npz', **output_dict)

In [ ]:
### CCMD 20Vars box, rsd, cov
vars = ['Var01', 'Var02', 'Var03', 'Var04', 'Var05', 'Var06', 'Var07', 'Var08', 'Var09', 'Var10', 
        'Var11', 'Var12', 'Var13', 'Var14', 'Var15', 'Var16', 'Var17', 'Var18', 'Var19', 'Var20', ]

output_dict_stat = collections.defaultdict(list)
output_dict_cov  = collections.defaultdict(list)

for j in range(4): # Mr, z
    data  = []
    for var in vars:
        temp = np.load('./CCMD_20Vars_Mrp/CCMD_' + var  + '_box_rsd_Mr_bins_Mrp_pimax60_60_60_40.npz')
        data.append(temp['stat_' + Mr_thr_list[j]])

    data  = np.array(data)

    output_dict_stat['stat_' + Mr_thr_list[j] ] = np.mean(data, axis=0)
    output_dict_cov[ 'cov_'  + Mr_thr_list[j] ] = np.cov(data, rowvar=False) 

    np.savez('./CCMD_20Vars_Mrp/CCMD_20Vars_box_rsd_Mr_bins_Mrp_pimax_60_60_60_40_stat_mean.npz', **output_dict_stat)
    np.savez('./CCMD_20Vars_Mrp/CCMD_20Vars_box_rsd_Mr_bins_Mrp_pimax_60_60_60_40_cov.npz', **output_dict_cov)

### Plot

In [ ]:
from matplotlib import cm

colors  = [cm.YlGnBu(55)  , cm.YlGnBu(80)  , cm.YlGnBu(130) , cm.YlGnBu(180) , cm.YlGnBu(220)]

In [ ]:
skibba2006_Mrp_mean = np.loadtxt('/home/wenlk/AssemblyBias/code/CCMD_color_mag_bin/Skibba2006_Mrp_mean.csv', delimiter=',', dtype=float)
skibba2006_Mrp_min  = np.loadtxt('/home/wenlk/AssemblyBias/code/CCMD_color_mag_bin/Skibba2006_Mrp_min.csv' , delimiter=',', dtype=float)
skibba2006_Mrp_max  = np.loadtxt('/home/wenlk/AssemblyBias/code/CCMD_color_mag_bin/Skibba2006_Mrp_max.csv' , delimiter=',', dtype=float)

yerr1 = skibba2006_Mrp_mean[:, 1] - skibba2006_Mrp_min[:, 1]
yerr2 = skibba2006_Mrp_max[:, 1]  - skibba2006_Mrp_mean[:, 1]

In [ ]:

Mrp_SDSS_bright1 = np.load('../CCMD_mag_bin/Mr_wp/SDSS_bright1_Mrp_pimax60_Mr_bins.npz')
Mrp_SDSS_bright1_cov = np.load('../CCMD_mag_bin/Mr_wp/SDSS_bright1_Mrp_cov_pimax60_Mr_bins.npz')
Mr1918_Mrp_SDSS_bright1 = np.load('../CCMD_mag_bin/Mr_wp/SDSS_bright1_Mrp_pimax40_Mr1918_bins.npz')
Mr1918_Mrp_SDSS_bright1_cov = np.load('../CCMD_mag_bin/Mr_wp/SDSS_bright1_Mrp_cov_pimax40_Mr1918_bins.npz')

CCMD_32Quads_stat = np.load('./CCMD_32Quads_Mr_bins_Mrp_pimax_60_60_60_40_stat_mean.npz')
CCMD_32Quads_cov  = np.load('./CCMD_32Quads_Mr_bins_Mrp_pimax_60_60_60_40_cov.npz')

In [ ]:
CCMD_20Vars_stat = np.load('./CCMD_20Vars_Mrp/CCMD_20Vars_box_rsd_Mr_bins_Mrp_pimax_60_60_60_40_stat_mean.npz')
CCMD_20Vars_cov  = np.load('./CCMD_20Vars_Mrp/CCMD_20Vars_box_rsd_Mr_bins_Mrp_pimax_60_60_60_40_cov.npz')

In [ ]:
fig, ax = plt.subplots(1, 4, figsize = (20, 5), sharex = True, sharey = True)
plt.subplots_adjust(hspace = 0, wspace = 0)

cov_text        = ['covMr_22_21', 'covMr_21_20', 'covMr_20_19', 'covMr_19_18']
pimax_texts     = [r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 40$']

### legend ####################################################################################
j = 0
ax[j].plot( rp_bins_wp_x, CCMD_32Quads_stat['stat_' + Mr_thr_list[j]][1:13], 
                color = 'forestgreen', alpha = 0.9, lw = 3, ls = '-', label = 'CCMD')
ax[j].errorbar( rp_bins_wp_x, Mrp_SDSS_bright1['stat_' + Mr_thr_list[j]][1:13],
                yerr = np.sqrt(Mrp_SDSS_bright1_cov['cov' + Mr_thr_list[j]].diagonal())[1:13],
                color = 'orange', lw = 2.5, alpha = 1, ls = 'none', capsize = 0, 
                marker = 'o', ms = 6, mec = 'orange', mew = 2.5, mfc = 'white', label = 'SDSS')
# ax[1].errorbar(skibba2006_Mrp_mean[:, 0], skibba2006_Mrp_mean[:, 1], yerr = yerr1, color = 'purple',
#                marker = 'o', ls = 'none', mfc = 'white', capsize = 0, lw = 2.5, ms = 5, mew = 2.5,
#                label = 'Skibba+2006, Mr < -20.5')
################################################################################################
for j in range(4):
### CCMD Quads
    ax[j].axhline(y=1, color='lightgray', linestyle='--')

    stat = CCMD_32Quads_stat['stat_' + Mr_thr_list[j]][1:13]
    ax[j].plot(rp_bins_wp_x, stat, color = 'forestgreen', alpha = 0.9, lw = 3, ls = '-')
    cov  = CCMD_32Quads_cov['cov_' + Mr_thr_list[j]]
    yerr = np.sqrt(cov.diagonal())[1:13]
    ax[j].fill_between( rp_bins_wp_x, stat - yerr, stat + yerr, color = 'forestgreen', alpha = 0.3,)
    cov_20Vars = CCMD_20Vars_cov[ 'cov_'  + Mr_thr_list[j]]
    yerr_total = np.sqrt((cov + cov_20Vars).diagonal())[1:13]
    ax[j].fill_between(rp_bins_wp_x, stat - yerr_total, stat - yerr, color = 'royalblue'  , alpha = 0.3,) # add CCMD 20Vars box err
    ax[j].fill_between(rp_bins_wp_x, stat + yerr, stat + yerr_total, color = 'royalblue'  , alpha = 0.3,) # add CCMD 20Vars box err

# ### CCMD 20 Vars box, rsd #######################################################################
#     stat = CCMD_20Vars_stat['stat_' + Mr_thr_list[j]][1:13]  
#     yerr = np.sqrt(CCMD_20Vars_cov[ 'cov_'  + Mr_thr_list[j]].diagonal())[1:13]
#     ax[j].plot(rp_bins_wp_x, stat, color = 'royalblue', alpha = 0.9, lw = 3, ls = '-', label = 'CCMD 20vars box, rsd')
#     ax[j].fill_between( rp_bins_wp_x, stat - yerr, stat + yerr, color = 'royalblue', alpha = 0.3)

for j in range(3):
### SDSS
    ax[j].errorbar( rp_bins_wp_x, Mrp_SDSS_bright1['stat_' + Mr_thr_list[j]][1:13],
                    yerr = np.sqrt(Mrp_SDSS_bright1_cov['cov' + Mr_thr_list[j]].diagonal())[1:13],
                    color = 'orange', lw = 2.5, alpha = 1, ls = 'none', capsize = 0, 
                    marker = 'o', ms = 6, mec = 'orange', mew = 2.5, mfc = 'white')
################################################################################################
for j in [3]:
    ### SDSS
    ax[j].errorbar( rp_bins_wp_x, Mr1918_Mrp_SDSS_bright1['stat_' + Mr_thr_list[j]][1:13],
                    yerr = np.sqrt(Mrp_SDSS_bright1_cov['cov' + Mr_thr_list[j]].diagonal())[1:13],
                    color = 'orange', lw = 2.5, alpha = 1, ls = 'none', capsize = 0, 
                    marker = 'o', ms = 6, mec = 'orange', mew = 2.5, mfc = 'white',)
################################################################################################

for j in range(4):
### format
    ax[j].set_xscale('log')
    ax[j].tick_params(axis = 'x', direction = 'in', which = u'both')
    ax[j].tick_params(axis = 'y', direction = 'in', which = u'both')
    ax[j].tick_params(axis='both', which='major', width=1.5, length=6)
    ax[j].tick_params(axis='both', which='minor', width=1.0, length=4)
    ax[j].text(0.24,0.92, Mr_thr_text[j], fontsize = 16)
    # ax[j].text(4  ,0.9, pimax_texts[j], fontsize = 16)
#     # ax[i].set_xlim(0.15, 40)
    ax[j].set_ylim(0.85, 1.75)
    ax[0].set_ylabel(r'$M(r_p)$',fontsize = 24,)
    # ax[j].set_xlabel(r'$r_p\ [h^{-1}{\rm Mpc}]$',fontsize = 16,)

legend = ax[0].legend(fontsize = 16, loc = 'upper right',frameon = False) 
legend.get_texts()[0].set_color('forestgreen')
# legend.get_texts()[1].set_color('royalblue')
legend.get_texts()[1].set_color('orange')


fig.supxlabel(r'$r_p\ [h^{-1}{\rm Mpc}]$', fontsize=24, fontweight='bold', y = -0.06)

# legend = ax[1].legend(fontsize = 16, loc = 'upper left',frameon = False) 
# legend.get_texts()[0].set_color('purple')

# plt.suptitle('(1 + Wp_rp/rp)/(1 + wp_rp/rp), weight = gmr/gmr_mean_in_Mr_bins', fontsize = 16, y = 0.98 )
# plt.savefig('../../../Figures/fig_data_Mr_C_m2pcf.pdf', bbox_inches='tight')

In [ ]:
wp_rp_red_blue_CCMD_32Quads     = np.load('./CCMD_32Quads_Mr_redest_bluest_cross_wprp_pimax_60_60_60_40_stat_mean.npz')
wp_rp_red_blue_CCMD_32Quads_cov = np.load('./CCMD_32Quads_Mr_redest_bluest_cross_wprp_pimax_60_60_60_40_cov.npz')

wp_rp_red_blue_SDSS_bright1        = np.load('../CCMD_mag_bin/Cross_correlation/SDSS_bright1_Mr_red_blue_cross_wprp_pimax60.npz')
Mr1918_wp_rp_red_blue_SDSS_bright1 = np.load('../CCMD_mag_bin/Cross_correlation/SDSS_bright1_Mr1918_red_blue_cross_wprp_pimax40.npz')
wp_rp_red_blue_SDSS_bright1_cov    = np.load('../CCMD_mag_bin/Cross_correlation/SDSS_bright1_Mr_redest_bluest_cross_wprp_pimax60_60_60_40_cov.npz')

In [ ]:
z_max_arr  = np.array((       47650, 31900, 19250, 12500))/ c
M_thr_arr  = np.array((-22.0, -21.0, -20.0, -19.0, -18.0))
C_thr_arr  = np.array([[0.0,  0.82,  0.94,  0.97,   1.2,],
                       [0.0,  0.72,  0.90,  0.94,   1.2,],
                       [0.0,  0.55,  0.85,  0.92,   1.2,],
                       [0.0,  0.50,  0.73,  0.87,   1.2,],])

M_thr_list  = [ 'Mr_22_21', 'Mr_21_20', 'Mr_20_19', 'Mr_19_18', ]
C_thr_list  = np.array([['0_p82'  , 'p82_p94', 'p94_p97', 'p97_1p2',],
                        ['0_p72'  , 'p72_p90', 'p90_p94', 'p94_1p2',],
                        ['0_p55'  , 'p55_p85', 'p85_p92', 'p92_1p2',],
                        ['0_p50'  , 'p50_p73', 'p73_p87', 'p87_1p2',],])

Mr_thr_text = [r'$M_{\rm r}: [-22,-21]$',r'$M_{\rm r}: [-21,-20]$',r'$M_{\rm r}: [-20,-19]$',r'$M_{\rm r}: [-19,-18]$',]
Co_thr_text = np.array([[r'g-r: [0,0.82]', r'g-r: [0.82,0.94]', r'g-r: [0.94,0.97]', r'g-r: [0.97,1.2]',],
                        [r'g-r: [0,0.72]', r'g-r: [0.72,0.90]', r'g-r: [0.90,0.94]', r'g-r: [0.94,1.2]',],
                        [r'g-r: [0,0.55]', r'g-r: [0.55,0.85]', r'g-r: [0.85,0.92]', r'g-r: [0.92,1.2]',],
                        [r'g-r: [0,0.50]', r'g-r: [0.50,0.73]', r'g-r: [0.73,0.87]', r'g-r: [0.87,1.2]',],])

Mr_cross_list = ['2221_2120', '2221_2019', '2221_1918', '2120_2019']

In [ ]:
wp_rp_red_blue_CCMD_20Vars     = np.load('./CCMD_20Vars_cross_2pcf/CCMD_20Vars_Mr_redest_bluest_cross_wprp_pimax_60_60_60_40_stat_mean.npz')
wp_rp_red_blue_CCMD_20Vars_cov = np.load('./CCMD_20Vars_cross_2pcf/CCMD_20Vars_Mr_redest_bluest_cross_wprp_pimax_60_60_60_40_cov.npz')

In [ ]:
pimax_thr_text = [r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 40$']
# color_cross_text = [r'$[0, 0.82]\times[0.97, 1.2]$', r'$[0, 0.72]\times[0.94, 1.2]$', r'$[0, 0.55]\times[0.92, 1.2]$', r'$[0, 0.50]\times[0.87, 1.2]$']
color_cross_text = [r'$\times$', r'$\times$', r'$\times$', r'$\times$']
color_blue_text = [r'$[0, 0.82]$', r'$[0, 0.72]$', r'$[0, 0.55]$', r'$[0, 0.50]$']
color_red_text = [r'$[0.97, 1.2]$', r'$[0.94, 1.2]$', r'$[0.92, 1.2]$', r'$[0.87, 1.2]$']

fig, ax = plt.subplots(2, 4, figsize = (20, 8), sharex = True)
plt.subplots_adjust(hspace = 0, wspace = 0)

cov_text        = ['covMr_22_21', 'covMr_21_20', 'covMr_20_19', 'covMr_19_18']
pimax_texts     = [r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 40$']

### legend ####################################################################################
j = 0
ax[0, j].plot( rp_bins_wp_x, CCMD_32Quads_stat['stat_' + Mr_thr_list[j]][1:13], 
                color = 'forestgreen', alpha = 0.9, lw = 3, ls = '-', label = 'CCMD')
ax[0, j].errorbar( rp_bins_wp_x, Mrp_SDSS_bright1['stat_' + Mr_thr_list[j]][1:13],
                yerr = np.sqrt(Mrp_SDSS_bright1_cov['cov' + Mr_thr_list[j]].diagonal())[1:13],
                color = 'orange', lw = 2.5, alpha = 1, ls = 'none', capsize = 0, 
                marker = 'o', ms = 6, mec = 'orange', mew = 2.5, mfc = 'white', label = 'SDSS')
# ax[1].errorbar(skibba2006_Mrp_mean[:, 0], skibba2006_Mrp_mean[:, 1], yerr = yerr1, color = 'purple',
#                marker = 'o', ls = 'none', mfc = 'white', capsize = 0, lw = 2.5, ms = 5, mew = 2.5,
#                label = 'Skibba+2006, Mr < -20.5')
################################################################################################
for j in range(4):
### CCMD Quads
    ax[0, j].axhline(y=1, color='lightgray', linestyle='--')
    
    stat = CCMD_32Quads_stat['stat_' + Mr_thr_list[j]][1:13]
    ax[0, j].plot(rp_bins_wp_x, stat, color = 'forestgreen', alpha = 0.9, lw = 3, ls = '-')
    cov  = CCMD_32Quads_cov['cov_' + Mr_thr_list[j]]
    yerr = np.sqrt(cov.diagonal())[1:13]
    ax[0, j].fill_between( rp_bins_wp_x, stat - yerr, stat + yerr, color = 'forestgreen', alpha = 0.3,)
    cov_20Vars = CCMD_20Vars_cov[ 'cov_'  + Mr_thr_list[j]]
    yerr_total = np.sqrt((cov + cov_20Vars).diagonal())[1:13]
    ax[0, j].fill_between(rp_bins_wp_x, stat - yerr_total, stat - yerr, color = 'royalblue'  , alpha = 0.2,) # add CCMD 20Vars box err
    ax[0, j].fill_between(rp_bins_wp_x, stat + yerr, stat + yerr_total, color = 'royalblue'  , alpha = 0.2,) # add CCMD 20Vars box err

for j in range(3):
### SDSS
    ax[0, j].errorbar( rp_bins_wp_x, Mrp_SDSS_bright1['stat_' + Mr_thr_list[j]][1:13],
                    yerr = np.sqrt(Mrp_SDSS_bright1_cov['cov' + Mr_thr_list[j]].diagonal())[1:13],
                    color = 'orange', lw = 2.5, alpha = 1, ls = 'none', capsize = 0, 
                    marker = 'o', ms = 6, mec = 'orange', mew = 2.5, mfc = 'white')
################################################################################################
for j in [3]:
    ### SDSS
    ax[0, j].errorbar( rp_bins_wp_x, Mr1918_Mrp_SDSS_bright1['stat_' + Mr_thr_list[j]][1:13],
                    yerr = np.sqrt(Mrp_SDSS_bright1_cov['cov' + Mr_thr_list[j]].diagonal())[1:13],
                    color = 'orange', lw = 2.5, alpha = 1, ls = 'none', capsize = 0, 
                    marker = 'o', ms = 6, mec = 'orange', mew = 2.5, mfc = 'white',)
################################################################################################

# cross-correlation
#############################################################################################################   
for j in range(4):
    stat = wp_rp_red_blue_CCMD_32Quads['stat_' + Mr_cross_list[j]]
    cov  = wp_rp_red_blue_CCMD_32Quads_cov['cov_'  + Mr_cross_list[j]]
    yerr = np.sqrt( cov.diagonal())
    ax[1, j].plot( rp_bins_wp_x, stat, color = 'forestgreen', alpha = 0.9, lw = 3, ls = '-', label = 'CCMD')
    ax[1, j].fill_between( rp_bins_wp_x, stat - yerr, stat + yerr, color = 'forestgreen', alpha = 0.3, lw = 3, ls = '-')
    cov_20Vars = wp_rp_red_blue_CCMD_20Vars_cov['cov_' + Mr_cross_list[j]]
    yerr_total = np.sqrt((cov + cov_20Vars).diagonal())
    ax[1, j].fill_between(rp_bins_wp_x, stat - yerr_total, stat - yerr, color = 'royalblue'  , alpha = 0.2,) # add CCMD 20Vars box err
    ax[1, j].fill_between(rp_bins_wp_x, stat + yerr, stat + yerr_total, color = 'royalblue'  , alpha = 0.2,) # add CCMD 20Vars box err

for j in range(3):
    yerr = np.sqrt(wp_rp_red_blue_SDSS_bright1_cov['cov_' + M_thr_list[j]].diagonal())
    ax[1, j].errorbar(rp_bins_wp_x, wp_rp_red_blue_SDSS_bright1['wprp_red_blue_' + M_thr_list[j]], yerr = yerr,
                   color = 'orange', alpha = 0.9, mew = 3, marker='o', mfc = 'white',ls='none', label = 'SDSS')
#############################################################################################################   
for j in [3]:
    yerr = np.sqrt(wp_rp_red_blue_SDSS_bright1_cov['cov_' + M_thr_list[j]].diagonal())
    ax[1, j].errorbar(rp_bins_wp_x, Mr1918_wp_rp_red_blue_SDSS_bright1['wprp_red_blue_' + M_thr_list[j]], yerr = yerr,
                   color = 'orange', alpha = 0.9, mew = 3, marker='o', mfc = 'white',ls='none', label = 'SDSS')
#############################################################################################################  
for j in range(4): 
    ax[0, j].set_xscale('log')
    ax[0, j].tick_params(axis = 'x', direction = 'in', which = u'both')
    ax[0, j].tick_params(axis = 'y', direction = 'in', which = u'both')
    ax[0, j].tick_params(axis='both', which='major', width=1.5, length=6)
    ax[0, j].tick_params(axis='both', which='minor', width=1.0, length=4)
    ax[0, j].set_ylim(0.85, 1.75)
    if j != 0:
        ax[0, j].set_yticklabels([]) 
    else:
        ax[0, j].set_ylabel(r'$M(r_p)$',fontsize = 24,)

    # ax[0, j].set_xlabel(r'$r_p\ [h^{-1}{\rm Mpc}]$',fontsize = 16,)

    ax[1, j].loglog()
    ax[1, j].tick_params(axis = 'x', direction = 'in', which = u'both')
    ax[1, j].tick_params(axis = 'y', direction = 'in', which = u'both')
    ax[1, j].tick_params(axis='both', which='major', width=1.5, length=6)
    ax[1, j].tick_params(axis='both', which='minor', width=1.0, length=4)
    # ax[1, j].text(0.22, 0.9, Mr_thr_text[j]   , fontsize = 16)
    ax[0, j].set_title(Mr_thr_text[j]   , fontsize = 16)
    # ax[j].text(7.5, 120, pimax_thr_text[j], fontsize = 16)
    # ax[j].text(1.2, 120, color_cross_text[j], fontsize = 16)

    ax[1, j].text(0.65, 220, r'g-r:', fontsize = 16)
    ax[1, j].text(4.7, 220, color_cross_text[j], fontsize = 16)
    ax[1, j].text(1.2, 220, color_blue_text[j], fontsize = 16, color = 'royalblue')
    ax[1, j].text(6.6, 220, color_red_text[j], fontsize = 16, color = 'red')

#     # ax[i].set_xlim(0.15, 40)
    ax[1, j].set_ylim(0.5, 500)
    # ax[1, j].set_xlabel(r'$r_{\rm p}\ [h^{-1}{\rm Mpc}]$',fontsize = 16,)
    if j != 0:
        ax[1, j].set_yticklabels([]) 
    else:
        ax[1, j].set_ylabel(r'$w_{\rm p}(r_{\rm p})\ [h^{-1}{\rm Mpc}]$',fontsize = 24,)

#############################################################################################################  

legend = ax[0,0].legend(fontsize = 16, loc = 'upper left',frameon = False) 
legend.get_texts()[0].set_color('forestgreen')
legend.get_texts()[1].set_color('orange')

fig.supxlabel(r'$r_p\ [h^{-1}{\rm Mpc}]$', fontsize=24, fontweight='bold', y = 0.0)

# legend = ax[1].legend(fontsize = 16, loc = 'upper left',frameon = False) 
# legend.get_texts()[0].set_color('purple')

# plt.suptitle('(1 + Wp_rp/rp)/(1 + wp_rp/rp), weight = gmr/gmr_mean_in_Mr_bins', fontsize = 16, y = 0.98 )
# plt.savefig('../../../Figures/fig_data_Mr_C_m2pcf.pdf', bbox_inches='tight')

In [ ]:
pimax_thr_text = [r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 40$']
# color_cross_text = [r'$[0, 0.82]\times[0.97, 1.2]$', r'$[0, 0.72]\times[0.94, 1.2]$', r'$[0, 0.55]\times[0.92, 1.2]$', r'$[0, 0.50]\times[0.87, 1.2]$']
color_cross_text = [r'$\times$', r'$\times$', r'$\times$', r'$\times$']
color_blue_text = [r'$[0, 0.82]$', r'$[0, 0.72]$', r'$[0, 0.55]$', r'$[0, 0.50]$']
color_red_text = [r'$[0.97, 1.2]$', r'$[0.94, 1.2]$', r'$[0.92, 1.2]$', r'$[0.87, 1.2]$']
cov_text        = ['covMr_22_21', 'covMr_21_20', 'covMr_20_19', 'covMr_19_18']
pimax_texts     = [r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 40$']

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(20, 8), dpi = 150)

# 创建主GridSpec：2行（每行包含原图+比值图），4列
gs_main = gridspec.GridSpec(2, 4, figure=fig, height_ratios=[1, 1], hspace=0.05, wspace=0)

ax_main = [[None for _ in range(4)] for _ in range(2)]  # 原图
ax_ratio = [[None for _ in range(4)] for _ in range(2)]  # 比值图

percent = 100

for i in range(2):
    for j in range(4):
        # 为每个单元格创建嵌套的GridSpec (2行1列，高度比例2:1)
        gs_cell = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=gs_main[i, j], 
                                                  height_ratios=[2, 1], hspace=0)
        #### Marked 2pcf ##########################################################################################################
        # 原图（上部分）
        ax_main[i][j]  = fig.add_subplot(gs_cell[0])
        # 比值图（下部分）
        ax_ratio[i][j] = fig.add_subplot(gs_cell[1])

for j in range(4):
    # 原图（上部分）
    ### CCMD ###
    ax_main[0][j].axhline(y=1, color='lightgray', linestyle='--')
    
    stat = CCMD_32Quads_stat['stat_' + Mr_thr_list[j]][1:13]
    ax_main[0][j].plot(rp_bins_wp_x, stat, color = 'forestgreen', alpha = 0.9, lw = 3, ls = '-', label = 'CCMD')
    cov  = CCMD_32Quads_cov['cov_' + Mr_thr_list[j]]
    yerr = np.sqrt(cov.diagonal())[1:13]
    ax_main[0][j].fill_between( rp_bins_wp_x, stat - yerr, stat + yerr, color = 'forestgreen', alpha = 0.3,)
    cov_20Vars = CCMD_20Vars_cov[ 'cov_'  + Mr_thr_list[j]]
    yerr_total = np.sqrt((cov + cov_20Vars).diagonal())[1:13]
    ax_main[0][j].fill_between(rp_bins_wp_x, stat - yerr_total, stat - yerr, color = 'royalblue'  , alpha = 0.2,)
    ax_main[0][j].fill_between(rp_bins_wp_x, stat + yerr, stat + yerr_total, color = 'royalblue'  , alpha = 0.2,)
    ### SDSS ###
    if j == 3:
        stat_SDSS_Mr1918 = Mr1918_Mrp_SDSS_bright1['stat_' + Mr_thr_list[j]][1:13]
        yerr_SDSS_Mr1918 = np.sqrt(Mrp_SDSS_bright1_cov['cov' + Mr_thr_list[j]].diagonal())[1:13]
        ax_main[0][j].errorbar( rp_bins_wp_x, stat_SDSS_Mr1918, yerr = yerr_SDSS_Mr1918,
                        color = 'orange', lw = 2.5, alpha = 1, ls = 'none', capsize = 0, 
                        marker = 'o', ms = 6, mec = 'orange', mew = 2.5, mfc = 'white', label = 'SDSS')
    else:
        stat_SDSS = Mrp_SDSS_bright1['stat_' + Mr_thr_list[j]][1:13]
        yerr_SDSS = np.sqrt(Mrp_SDSS_bright1_cov['cov' + Mr_thr_list[j]].diagonal())[1:13]
        ax_main[0][j].errorbar( rp_bins_wp_x, stat_SDSS, yerr = yerr_SDSS,
                        color = 'orange', lw = 2.5, alpha = 1, ls = 'none', capsize = 0, 
                        marker = 'o', ms = 6, mec = 'orange', mew = 2.5, mfc = 'white', label = 'SDSS')
    # 比值图（下部分）
    ### CCMD ###
    ax_ratio[0][j].plot(rp_bins_wp_x, (stat/stat-1)*percent, color = 'forestgreen', alpha = 0.9, lw = 3, ls = '-', label = 'CCMD')
    ax_ratio[0][j].fill_between(rp_bins_wp_x, ((stat - yerr)/stat-1)*percent, ((stat + yerr)/stat-1)*percent, color = 'forestgreen', alpha = 0.3,)
    ax_ratio[0][j].fill_between(rp_bins_wp_x, ((stat - yerr_total)/stat-1)*percent, ((stat - yerr)/stat-1)*percent, color = 'royalblue', alpha = 0.2,) 
    ax_ratio[0][j].fill_between(rp_bins_wp_x, ((stat + yerr)/stat-1)*percent, ((stat + yerr_total)/stat-1)*percent, color = 'royalblue', alpha = 0.2,) 
    ### SDSS ###
    if j == 3:
        ax_ratio[0][j].errorbar( rp_bins_wp_x, (stat_SDSS_Mr1918/stat - 1)*percent, yerr = (yerr_SDSS_Mr1918/stat)*percent,
                        color = 'orange', lw = 2.5, alpha = 1, ls = 'none', capsize = 0, 
                        marker = 'o', ms = 6, mec = 'orange', mew = 2.5, mfc = 'white', label = 'SDSS')
    else:
        ax_ratio[0][j].errorbar( rp_bins_wp_x, (stat_SDSS/stat - 1)*percent, yerr = (yerr_SDSS/stat)*percent,
                        color = 'orange', lw = 2.5, alpha = 1, ls = 'none', capsize = 0, 
                        marker = 'o', ms = 6, mec = 'orange', mew = 2.5, mfc = 'white', label = 'SDSS')
    ### cross-correlation ###################################################################################################
    # 原图（上部分）
    ### CCMD ### 
    stat = wp_rp_red_blue_CCMD_32Quads['stat_' + Mr_cross_list[j]]
    cov  = wp_rp_red_blue_CCMD_32Quads_cov['cov_'  + Mr_cross_list[j]]
    yerr = np.sqrt( cov.diagonal())
    ax_main[1][j].plot( rp_bins_wp_x, stat, color = 'forestgreen', alpha = 0.9, lw = 3, ls = '-', label = 'CCMD')
    ax_main[1][j].fill_between( rp_bins_wp_x, stat - yerr, stat + yerr, color = 'forestgreen', alpha = 0.3, lw = 3, ls = '-')
    cov_20Vars = wp_rp_red_blue_CCMD_20Vars_cov['cov_' + Mr_cross_list[j]]
    yerr_total = np.sqrt((cov + cov_20Vars).diagonal())
    ax_main[1][j].fill_between(rp_bins_wp_x, stat - yerr_total, stat - yerr, color = 'royalblue'  , alpha = 0.2,)
    ax_main[1][j].fill_between(rp_bins_wp_x, stat + yerr, stat + yerr_total, color = 'royalblue'  , alpha = 0.2,) 
    ### SDSS ### 
    if j == 3:
        stat_SDSS_Mr1918 = Mr1918_wp_rp_red_blue_SDSS_bright1['wprp_red_blue_' + M_thr_list[j]]
        yerr_SDSS_Mr1918 = np.sqrt(wp_rp_red_blue_SDSS_bright1_cov['cov_' + M_thr_list[j]].diagonal())
        ax_main[1][j].errorbar(rp_bins_wp_x, stat_SDSS_Mr1918, yerr = yerr_SDSS_Mr1918,
                    color = 'orange', alpha = 0.9, mew = 3, marker='o', mfc = 'white',ls='none', label = 'SDSS')
    else:
        stat_SDSS = wp_rp_red_blue_SDSS_bright1['wprp_red_blue_' + M_thr_list[j]]
        yerr_SDSS = np.sqrt(wp_rp_red_blue_SDSS_bright1_cov['cov_' + M_thr_list[j]].diagonal())
        ax_main[1][j].errorbar(rp_bins_wp_x, stat_SDSS, yerr = yerr_SDSS,
                    color = 'orange', alpha = 0.9, mew = 3, marker='o', mfc = 'white',ls='none', label = 'SDSS')
    # 比值图（下部分）
    ### CCMD ###
    ax_ratio[1][j].plot(rp_bins_wp_x, (stat/stat-1)*percent, color = 'forestgreen', alpha = 0.9, lw = 3, ls = '-', label = 'CCMD')
    ax_ratio[1][j].fill_between(rp_bins_wp_x, ((stat - yerr)/stat-1)*percent, ((stat + yerr)/stat-1)*percent, color = 'forestgreen', alpha = 0.3,)
    ax_ratio[1][j].fill_between(rp_bins_wp_x, ((stat - yerr_total)/stat-1)*percent, ((stat - yerr)/stat-1)*percent, color = 'royalblue', alpha = 0.2,) 
    ax_ratio[1][j].fill_between(rp_bins_wp_x, ((stat + yerr)/stat-1)*percent, ((stat + yerr_total)/stat-1)*percent, color = 'royalblue', alpha = 0.2,) 
    ### SDSS ###
    if j == 3:
        ax_ratio[1][j].errorbar( rp_bins_wp_x, (stat_SDSS_Mr1918/stat - 1)*percent, yerr = (yerr_SDSS_Mr1918/stat)*percent,
                        color = 'orange', lw = 2.5, alpha = 1, ls = 'none', capsize = 0, 
                        marker = 'o', ms = 6, mec = 'orange', mew = 2.5, mfc = 'white', label = 'SDSS')
    else:
        ax_ratio[1][j].errorbar( rp_bins_wp_x, (stat_SDSS/stat - 1)*percent, yerr = (yerr_SDSS/stat)*percent,
                        color = 'orange', lw = 2.5, alpha = 1, ls = 'none', capsize = 0, 
                        marker = 'o', ms = 6, mec = 'orange', mew = 2.5, mfc = 'white', label = 'SDSS')
    ########################################################################################################################################

for i in range(2):
    for j in range(4):
        ### ratio = 1
        ax_ratio[i][j].plot([-5, 60], [0, 0], color = 'gray', alpha = 0.6)
        # ax_main[0][j].text(0.22, 700, Mr_thr_text[j], fontsize=20, fontweight='bold',)  
        ##########################################################################        
        # 调整主图四个边框的粗细
        for spine in ax_main[i][j].spines.values():
                # 对于主图 - 分别调整每个边框
                ax_main[i][j].spines['top'].set_linewidth(2)
                ax_main[i][j].spines['bottom'].set_linewidth(1) 
                ax_main[i][j].spines['left'].set_linewidth(2)
                ax_main[i][j].spines['right'].set_linewidth(2)
        for spine in ax_ratio[i][j].spines.values():
                # 对于比值图 - 分别调整每个边框
                ax_ratio[i][j].spines['top'].set_linewidth(1)
                ax_ratio[i][j].spines['bottom'].set_linewidth(2)
                ax_ratio[i][j].spines['left'].set_linewidth(2)
                ax_ratio[i][j].spines['right'].set_linewidth(2)

        ax_main[i][j].set_xscale('log')       
        ax_main[1][j].loglog()
        # ax_main[i][j].set_yticks([1, 3, 5], ['1', '3', '5'])
        ax_main[i][j].tick_params(axis = 'x', direction = 'in', which = u'both', )
        ax_main[i][j].tick_params(axis = 'y', direction = 'in', which = u'both', )
        ax_main[i][j].tick_params(axis='both', which='major', width=2, length=8, labelsize=20,)
        # ax_main[i][j].tick_params(axis='both', which='minor', width=2, length=5, labelsize=20,)
        ax_main[i][j].tick_params(axis='x', which='minor', width=2, length=4, labelsize=20) # X轴子刻度仍然显示
        ax_main[i][j].tick_params(axis='y', which='minor', length=0)  # Y轴子刻度隐藏，设置长度为0来隐藏
        # ax_main[i][j].xaxis.set_minor_locator(plt.MultipleLocator(0.5))  # x轴子刻度
        # ax_main[i][j].yaxis.set_minor_locator(plt.MultipleLocator(1))  # y轴子刻度
        ax_main[i][j].set_xlim(0.15, 40)
        ax_main[0][j].set_ylim(0.85, 1.8)
        ax_main[1][j].set_ylim(0.2, 500)

        ax_ratio[i][j].set_xscale('log')       
        ax_ratio[i][j].tick_params(axis = 'x', direction = 'in', which = u'both', )
        ax_ratio[i][j].tick_params(axis = 'y', direction = 'in', which = u'both', )
        ax_ratio[i][j].tick_params(axis='both', which='major', width=2, length=8, labelsize=20,)
        ax_ratio[i][j].tick_params(axis='both', which='minor', width=2, length=4, labelsize=20,)
        # ax_ratio[i][j].xaxis.set_minor_locator(plt.MultipleLocator(0.5))  # x轴子刻度
        # ax_ratio[i][j].yaxis.set_minor_locator(plt.MultipleLocator(0.5))  # y轴子刻度
        ax_ratio[i][j].set_xlim(0.15, 40)
        # ax_ratio[0][j].set_yticks([-0.1, 0.1], ['-0.1', '0.1'])
        # ax_ratio[0][j].set_ylim(-0.2, 0.2)
        # ax_ratio[1][j].set_yticks([-0.5, 0.5], ['-0.5', '0.5'])
        # ax_ratio[1][j].set_ylim(-0.7, 0.7)
        # ax_ratio[0][j].set_yticks([-10, 10], ['-10%', '10%'])
        # ax_ratio[0][j].set_ylim(-15, 15)
        ax_ratio[0][0].set_yticks([-1, 1], [ ])
        ax_ratio[0][0].set_ylim(-3, 3)
        ax_ratio[0][0].set_ylabel('Diff.' , fontsize = 18)
        ax_ratio[0][0].text(0.2,  1.0, ' 1%', fontsize = 14)
        # ax_ratio[0][0].text(0.2, -1.5, '-1%', fontsize = 12)

        ax_ratio[0][1].set_yticks([-1, 1], ['-1%', '1%'])
        ax_ratio[0][1].set_ylim(-3, 3)
        ax_ratio[0][1].text(0.2,  1.0, '1%', fontsize = 14)

        ax_ratio[0][2].set_yticks([-5, 5], ['-5%', '5%'])
        ax_ratio[0][2].set_ylim(-7, 7)
        ax_ratio[0][2].text(0.2,  4, '5%', fontsize = 14)

        ax_ratio[0][3].set_yticks([-10, 10], ['-10%', '10%'])
        ax_ratio[0][3].set_ylim(-15, 15)
        ax_ratio[0][3].text(0.2,  8, '10%', fontsize = 14)

        ax_ratio[1][0].text(0.2, 10, 'Diff.', fontsize = 18)
        ax_ratio[1][j].set_yticks([-20, 20], ['-20%', '20%'])
        ax_ratio[1][j].set_ylim(-30, 30)

            
for i in range(2):
    for j in [1, 2, 3]:
        ax_main[i][j].tick_params( axis='both', which='major', labelsize=20, labelbottom=False, labelleft=False)
        ax_main[i][j].tick_params( axis='both', which='minor', labelsize=20, labelbottom=False, labelleft=False)
        ax_ratio[0][j].tick_params(axis='both', which='major', labelsize=20, labelbottom=False, labelleft=False)
        ax_ratio[0][j].tick_params(axis='both', which='minor', labelsize=20, labelbottom=False, labelleft=False)
        ax_ratio[0][0].tick_params(axis='both', which='major', labelsize=20, labelbottom=False)
        ax_ratio[0][0].tick_params(axis='both', which='minor', labelsize=20, labelbottom=False)
        ax_ratio[1][j].tick_params(axis='both', which='major', labelsize=20, labelleft=False)
        ax_ratio[1][j].tick_params(axis='both', which='minor', labelsize=20, labelleft=False)
     
for j in range(4):
    ax_main[0][j].text(1.8, 1.62, Mr_thr_text[j], fontsize=18, fontweight='bold',)
    ax_main[0][0].set_ylabel(r'$M(r_p)$',fontsize = 24,)
    ax_main[1][0].text(2 , 130, r'blue'    , fontsize = 20, color = 'royalblue')
    ax_main[1][0].text(5 , 130, r'$\times$', fontsize = 20)
    ax_main[1][0].text(8 , 130, r'redder ' , fontsize = 20, color = 'red')


    # ax_main[1][j].text(0.6 , 150, r'g-r:', fontsize = 16)
    # ax_main[1][j].text(4.65, 150, color_cross_text[j], fontsize = 16)
    # ax_main[1][j].text(1.15, 150, color_blue_text[j], fontsize = 16, color = 'royalblue')
    # ax_main[1][j].text(6.55, 150, color_red_text[j], fontsize = 16, color = 'red')

    if j != 0:
        ax_main[1][j].set_yticklabels([]) 
    else:
        ax_main[1][j].set_ylabel(r'$w_{\rm p}(r_{\rm p})$', fontsize = 24,)

    for i in range(2):
        ax_ratio[i][j].tick_params(axis = 'y', direction = 'in', which = u'both', labelsize = 14)

#############################################################################################################  

legend = ax_main[0][0].legend(fontsize = 16, loc = 'upper left',frameon = False) 
legend.get_texts()[0].set_color('forestgreen')
legend.get_texts()[1].set_color('orange')

fig.supxlabel(r'$r_p\ [h^{-1}{\rm Mpc}]$', fontsize=24, fontweight='bold', y = 0.0)

plt.tight_layout()
plt.show()

In [ ]:
sdss_bright1 = fits.open('../SDSS_bright1_processed_catalog.fits')[1].data
# sdss_bright1
bright1_ra  = sdss_bright1['ra'] # in degrees
bright1_dec = sdss_bright1['dec'] # in degrees
bright1_z   = sdss_bright1['z']
bright1_Mr  = sdss_bright1['Mr'] 

np.max(bright1_z[bright1_Mr > -20])

In [ ]:
data = fits.open('../CCMD_main_processed/CCMD_32Quads_processed/CCMD_' + vars[0] + '_' + quads[0] + '_processed_catalog.fits')[1].data 
plt.hist(data['z'][data['Mr'] > -20], density=True, color = 'green' , histtype='step' ,label = 'CCMD')
plt.hist(bright1_z[bright1_Mr > -20], density=True, color = 'orange', histtype='step' ,label = 'SDSS bright1')
plt.legend()
